# NeuroState multi seed runs

Camera ready statistical rigor pass for MLSP 2026.

Sleep-EDF holds the subject split fixed at `split_seed=42` so that variance across seeds reflects initialisation and batch order on the same test subjects as the submitted results. CHB-MIT uses the round robin split, which is deterministic and therefore fixed by construction.

Every seed is checkpointed to JSON as it finishes, so a Deepnote restart never loses completed work. Re running a cell resumes from the seeds already present.

In [1]:
# IMPORTS

import json
import math
import random
import copy
import time
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, cohen_kappa_score, roc_auc_score,
                             recall_score, confusion_matrix)
from tqdm.auto import tqdm
from scipy.stats import mannwhitneyu

print(torch.__version__, torch.cuda.is_available())

In [2]:
# MODEL AND DATA DEFINITIONS
#   PatchEmbedding, SingleScaleBoundaryEstimator,
#   MultiScaleChangepointModule, ContrastiveBoundaryModule,
#   RegimeStructuredAttention, MultiResolutionEncoder,
#   MultiResContrastiveNeuroState, MultiEpochDataset,
#   PseudoBoundaryLoss, AttentionPriorLoss, ACBLLoss,
#   forward_with_intermediates, create_acbl_dataloaders

# Unchanged from ACBL_and_Gradient_Isolation so the multi seed
# results correspond to the same architecture.

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=22, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)

class SingleScaleBoundaryEstimator(nn.Module):
    def __init__(self, embed_dim, hidden_dim, kernel_size, dropout=0.1):
        super().__init__()
        ks = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
        pad = ks // 2
        self.net = nn.Sequential(
            nn.Conv1d(embed_dim, hidden_dim, ks, padding=pad),
            nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, ks, padding=pad),
            nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, 1, 1))

    def forward(self, x):
        h = self.net(x.permute(0, 2, 1))
        return torch.sigmoid(h.squeeze(1))

class MultiScaleChangepointModule(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(3, 13, 65), dropout=0.1,
                 lambda_sparse=0.02, lambda_sharp=0.02):
        super().__init__()
        self.scales = scales
        self.n_scales = len(scales)
        self.lambda_sparse = lambda_sparse
        self.lambda_sharp = lambda_sharp
        self.estimators = nn.ModuleList([
            SingleScaleBoundaryEstimator(embed_dim, hidden_dim, k, dropout)
            for k in scales])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_scales, self.n_scales * 2), nn.GELU(),
            nn.Linear(self.n_scales * 2, 1))

    def forward(self, x):
        per_scale = [est(x) for est in self.estimators]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        boundary_loss = self._boundary_loss(fused, per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': boundary_loss}

    def _boundary_loss(self, fused, per_scale):
        sparsity = fused.mean()
        eps = 1e-7
        entropy = -(fused * torch.log(fused + eps) +
                    (1 - fused) * torch.log(1 - fused + eps))
        sharpness = entropy.mean()
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return (self.lambda_sparse * sparsity +
                self.lambda_sharp * sharpness + 0.01 * consistency)

class ContrastiveBoundaryModule(nn.Module):
    """Boundaries from representation contrast, not absolute MLP."""

    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.n_scales = len(scales)
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
            for _ in scales])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_scales, self.n_scales * 2), nn.GELU(),
            nn.Linear(self.n_scales * 2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def _compute_contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = proj(x)
        h_norm = F.normalize(h, dim=-1)
        if offset < T:
            h_shifted = torch.roll(h_norm, -offset, dims=1)
            h_shifted[:, -offset:, :] = h_norm[:, -offset:, :]
            similarity = (h_norm * h_shifted).sum(dim=-1)
            contrast = 1.0 - (similarity + 1.0) / 2.0
            contrast = torch.sigmoid(
                (contrast - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            contrast = torch.zeros(B, T, device=x.device)
        return contrast

    def forward(self, x):
        per_scale = [self._compute_contrast(x, proj, off)
                     for proj, off in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': 0.01 * consistency}

class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.n_cross = n_cross
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        assert embed_dim % self.n_heads == 0
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def _build_regime_mask(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        return torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))

    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        same = self._build_regime_mask(boundaries).unsqueeze(1)
        cross = 1.0 - same
        h_ie = self.n_intra
        h_ce = h_ie + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h_ie] = same.expand(B, self.n_intra, T, T)
        mask[:, h_ie:h_ce] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)
        attn_w = F.softmax(attn, dim=-1)
        attn_w = self.attn_drop(attn_w)
        out = (attn_w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))
        return (out, attn_w) if return_attention else out

class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(
            embed_dim, n_intra, n_inter, n_cross, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout))

    def forward(self, x, boundaries, return_attention=False):
        normed = self.norm1(x)
        if return_attention:
            a, w = self.attn(normed, boundaries, True)
            x = x + a
            x = x + self.mlp(self.norm2(x))
            return x, w
        x = x + self.attn(normed, boundaries)
        x = x + self.mlp(self.norm2(x))
        return x

class MultiResolutionEncoder(nn.Module):
    """Encodes a single epoch at 100/50/25 Hz, merges via interpolation."""

    def __init__(self, n_channels=3, n_samples=3000,
                 embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100hz = PatchEmbedding(
            n_channels, n_samples, embed_dim, dropout=dropout)
        self.enc_50hz = PatchEmbedding(
            n_channels, n_samples // 2, embed_dim, dropout=dropout)
        self.enc_25hz = PatchEmbedding(
            n_channels, n_samples // 4, embed_dim, dropout=dropout)
        self.merge = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100hz.seq_len
        self.seq_len_50 = self.enc_50hz.seq_len
        self.seq_len_25 = self.enc_25hz.seq_len

    def forward(self, x):
        x_50 = x[:, :, ::2]
        x_25 = x[:, :, ::4]
        emb_100 = self.enc_100hz(x)
        emb_50 = self.enc_50hz(x_50)
        emb_25 = self.enc_25hz(x_25)
        T1 = emb_100.shape[1]
        emb_50_up = F.interpolate(
            emb_50.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        emb_25_up = F.interpolate(
            emb_25.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        merged = torch.cat([emb_100, emb_50_up, emb_25_up], dim=-1)
        return self.merge(merged)

class MultiResContrastiveNeuroState(nn.Module):
    """MultiRes MultiEpoch NeuroState with contrastive boundaries."""

    def __init__(self, n_channels=3, n_samples=3000, n_classes=5,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 n_intra=4, n_inter=2, n_cross=2,
                 contrast_scales=(1, 4, 16), cp_hidden=64,
                 n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes
        self.n_context = n_context_epochs
        self.mr_encoder = MultiResolutionEncoder(
            n_channels, n_samples, embed_dim, dropout)
        tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = tokens_per_epoch * n_context_epochs
        self.pos_embed = nn.Parameter(
            torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(
            torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)
        self.changepoint_module = ContrastiveBoundaryModule(
            embed_dim=embed_dim, hidden_dim=cp_hidden,
            scales=contrast_scales, dropout=dropout)
        self.blocks = nn.ModuleList([
            NeuroStateBlock(embed_dim, n_intra, n_inter, n_cross,
                            dropout=dropout)
            for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, n_classes))
        self.tokens_per_epoch = tokens_per_epoch
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"ContrastiveMultiRes: {n_params/1e6:.2f}M params, "
              f"{total_tokens} tokens")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, return_boundaries=False):
        B, N, C, T = x.shape
        epoch_embs = []
        for i in range(N):
            emb = self.mr_encoder(x[:, i])
            emb = emb + self.epoch_embed[:, i]
            epoch_embs.append(emb)
        full_seq = torch.cat(epoch_embs, dim=1)
        full_seq = self.pos_drop(full_seq + self.pos_embed)
        cp_out = self.changepoint_module(full_seq)
        boundaries = cp_out['boundaries']
        boundary_loss = cp_out['boundary_loss']
        for block in self.blocks:
            full_seq = block(full_seq, boundaries)
        full_seq = self.norm(full_seq)
        start = self.tokens_per_epoch * (N // 2)
        end = start + self.tokens_per_epoch
        center_tokens = full_seq[:, start:end, :]
        pooled = center_tokens.mean(dim=1)
        logits = self.head(pooled)
        out = {'logits': logits, 'boundary_loss': boundary_loss}
        if return_boundaries:
            out['boundaries'] = boundaries
            out['per_scale'] = cp_out['per_scale']
        return out

class MultiEpochDataset(Dataset):
    """Returns 3 consecutive epochs. Label = center epoch."""

    def __init__(self, h5_path, task='sleep_staging',
                 target_channels=3, target_samples=3000,
                 indices=None, context_size=1):
        self.h5_path = Path(h5_path)
        self.target_channels = target_channels
        self.target_samples = target_samples
        self.context_size = context_size
        self._h5_file = None

        with h5py.File(self.h5_path, 'r') as f:
            self.total_samples = f['epochs'].shape[0]
            self.labels = f['labels'][:]
            self.subject_ids = f['subject_ids'][:]

        if isinstance(self.subject_ids[0], (bytes, np.bytes_)):
            self.subject_ids = np.array([
                s.decode() if isinstance(s, bytes) else s
                for s in self.subject_ids])

        base_indices = indices if indices is not None else np.arange(
            self.total_samples)
        base_set = set(base_indices)
        self.valid_indices = []

        for idx in base_indices:
            valid = True
            for offset in range(-context_size, context_size + 1):
                neighbor = idx + offset
                if neighbor < 0 or neighbor >= self.total_samples:
                    valid = False; break
                if neighbor not in base_set:
                    valid = False; break
                if self.subject_ids[neighbor] != self.subject_ids[idx]:
                    valid = False; break
            if valid:
                self.valid_indices.append(idx)

        self.valid_indices = np.array(self.valid_indices)
        self._compute_class_weights()
        dist = dict(zip(*np.unique(
            self.labels[self.valid_indices], return_counts=True)))
        print(f"  MultiEpoch: {len(self.valid_indices)} seqs, "
              f"classes={dist}")

    def _get_h5(self):
        if self._h5_file is None:
            self._h5_file = h5py.File(self.h5_path, 'r')
        return self._h5_file

    def _compute_class_weights(self):
        sub = self.labels[self.valid_indices]
        cls, cnt = np.unique(sub, return_counts=True)
        w = 1.0 / cnt; w /= w.sum()
        self.class_weights = dict(zip(cls, w))
        self.sample_weights = np.array(
            [self.class_weights[l] for l in sub])

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        center_idx = self.valid_indices[idx]
        f = self._get_h5()
        epochs = []
        epoch_labels = []
        for offset in range(-self.context_size,
                            self.context_size + 1):
            ep = f['epochs'][center_idx + offset]
            ep = self._standardize_shape(ep)
            epochs.append(ep)
            epoch_labels.append(self.labels[center_idx + offset])

        return {
            'epoch': torch.tensor(np.stack(epochs, axis=0),
                                  dtype=torch.float32),
            'label': torch.tensor(self.labels[center_idx],
                                  dtype=torch.long),
            'epoch_labels': torch.tensor(epoch_labels,
                                         dtype=torch.long),
        }

    def _standardize_shape(self, epoch):
        nc, nt = epoch.shape
        if nc < self.target_channels:
            epoch = np.vstack(
                [epoch, np.zeros((self.target_channels - nc, nt))])
        elif nc > self.target_channels:
            epoch = epoch[:self.target_channels]
        nc = self.target_channels
        if nt < self.target_samples:
            epoch = np.hstack(
                [epoch, np.zeros((nc, self.target_samples - nt))])
        elif nt > self.target_samples:
            s = (nt - self.target_samples) // 2
            epoch = epoch[:, s:s + self.target_samples]
        return epoch

    def get_sampler(self):
        return WeightedRandomSampler(
            self.sample_weights, len(self), True)

class PseudoBoundaryLoss(nn.Module):
    """
    Prong 1: Self supervised boundary signal from encoder representations.

    Computes cosine distance between adjacent tokens. High distance
    means representational shift, used as soft target for boundary head.
    Targets are detached to prevent encoder from trivially maximizing
    adjacent dissimilarity.
    """

    def __init__(self, temperature=2.0, tokens_per_epoch=196, n_epochs=3):
        super().__init__()
        self.temperature = temperature
        self.tokens_per_epoch = tokens_per_epoch
        self.n_epochs = n_epochs

        T = tokens_per_epoch * n_epochs
        mask = torch.ones(T - 1)
        for e in range(1, n_epochs):
            mask[e * tokens_per_epoch - 1] = 0.0
        self.register_buffer('epoch_boundary_mask', mask)

    def compute_pseudo_targets(self, h):
        """h: [B, 588, 128] encoder output. Returns [B, 588] soft targets."""
        B, T, D = h.shape
        h_det = h.detach()

        h_norm = F.normalize(h_det, dim=-1)
        cos_sim = (h_norm[:, :-1] * h_norm[:, 1:]).sum(dim=-1)
        cos_dist = (1.0 - cos_sim) * self.epoch_boundary_mask.to(h.device)

        d_min = cos_dist.min(dim=-1, keepdim=True).values
        d_max = cos_dist.max(dim=-1, keepdim=True).values
        d_range = (d_max - d_min).clamp(min=1e-6)
        cos_dist_norm = (cos_dist - d_min) / d_range

        pseudo = torch.sigmoid((cos_dist_norm - 0.5) * self.temperature)
        return F.pad(pseudo, (0, 1), mode='replicate')

    def forward(self, boundary_probs, h):
        pseudo_targets = self.compute_pseudo_targets(h)
        return F.binary_cross_entropy(
            boundary_probs.clamp(1e-6, 1 - 1e-6),
            pseudo_targets, reduction='mean')

class AttentionPriorLoss(nn.Module):
    """
    Prong 2: Label derived boundary and attention supervision.

    2A: Gaussian peaks at epoch junctions where labels change.
        Direct BCE on boundary head output.
    2B: KL divergence between regime attention and a block diagonal
        prior derived from epoch labels. Gives gradient through
        the attention mechanism back to the boundary head.
    """

    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.tokens_per_epoch = tokens_per_epoch
        self.n_epochs = n_epochs
        self.sigma = sigma
        self.attn_prior_weight = attn_prior_weight
        self.T = tokens_per_epoch * n_epochs

    def _build_boundary_targets(self, labels, device):
        B = labels.shape[0]
        T = self.T
        tpe = self.tokens_per_epoch

        trans_01 = (labels[:, 0] != labels[:, 1]).float()
        trans_12 = (labels[:, 1] != labels[:, 2]).float()
        has_transition = (trans_01 + trans_12) > 0

        positions = torch.arange(T, device=device, dtype=torch.float32)
        targets = torch.zeros(B, T, device=device)

        junction_01 = float(tpe) - 0.5
        junction_12 = float(2 * tpe) - 0.5

        gauss_01 = torch.exp(
            -0.5 * ((positions - junction_01) / self.sigma) ** 2)
        gauss_12 = torch.exp(
            -0.5 * ((positions - junction_12) / self.sigma) ** 2)

        targets += trans_01.unsqueeze(1) * gauss_01.unsqueeze(0)
        targets += trans_12.unsqueeze(1) * gauss_12.unsqueeze(0)

        t_max = targets.max(dim=-1, keepdim=True).values.clamp(min=1e-6)
        targets = targets / t_max
        targets = targets * has_transition.float().unsqueeze(1)

        return targets, has_transition

    def _build_attention_prior(self, labels, device):
        B = labels.shape[0]
        T = self.T
        tpe = self.tokens_per_epoch

        token_labels = torch.zeros(B, T, dtype=labels.dtype, device=device)
        for e in range(self.n_epochs):
            start = e * tpe
            end = (e + 1) * tpe
            token_labels[:, start:end] = labels[:, e].unsqueeze(1)

        same_stage = (token_labels.unsqueeze(2) ==
                      token_labels.unsqueeze(1)).float()
        prior = same_stage * 0.8 + (1 - same_stage) * 0.2
        prior = prior / prior.sum(dim=-1, keepdim=True)
        return prior

    def forward(self, boundary_probs, regime_attention, labels):
        device = boundary_probs.device
        boundary_targets, has_transition = self._build_boundary_targets(
            labels, device)
        n_transitions = has_transition.sum().item()

        results = {
            'n_transitions': n_transitions,
            'boundary_target_loss': torch.tensor(0.0, device=device),
            'attention_prior_loss': torch.tensor(0.0, device=device),
        }

        if n_transitions == 0:
            results['total'] = torch.tensor(0.0, device=device)
            return results

        mask = has_transition.float()
        bce = F.binary_cross_entropy(
            boundary_probs.clamp(1e-6, 1 - 1e-6),
            boundary_targets, reduction='none')
        bce_per_sample = bce.mean(dim=-1)
        results['boundary_target_loss'] = (
            (bce_per_sample * mask).sum() / mask.sum())

        if regime_attention is not None:
            if regime_attention.dim() == 4:
                attn = regime_attention.mean(dim=1)
            else:
                attn = regime_attention

            prior = self._build_attention_prior(labels, device)
            log_attn = torch.log(attn.clamp(min=1e-8))
            log_prior = torch.log(prior.clamp(min=1e-8))
            kl = prior * (log_prior - log_attn)
            kl_per_sample = kl.sum(dim=-1).mean(dim=-1)
            results['attention_prior_loss'] = (
                (kl_per_sample * mask).sum() / mask.sum())

        results['total'] = (
            results['boundary_target_loss'] +
            self.attn_prior_weight * results['attention_prior_loss'])
        return results

class ACBLLoss(nn.Module):
    """Combined ACBL loss with temperature annealing and variance penalty."""

    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 pseudo_weight=0.3, pseudo_temperature=2.0,
                 pseudo_temp_min=0.5, pseudo_temp_anneal_epochs=15,
                 prior_weight=0.5, sigma=5.0,
                 attn_prior_weight=0.5,
                 variance_weight=1.0):
        super().__init__()
        self.pseudo_weight = pseudo_weight
        self.prior_weight = prior_weight
        self.variance_weight = variance_weight
        self.pseudo_temp_min = pseudo_temp_min
        self.pseudo_temp_anneal_epochs = pseudo_temp_anneal_epochs
        self.pseudo_temperature_init = pseudo_temperature

        self.prong1 = PseudoBoundaryLoss(
            temperature=pseudo_temperature,
            tokens_per_epoch=tokens_per_epoch,
            n_epochs=n_epochs)

        self.prong2 = AttentionPriorLoss(
            tokens_per_epoch=tokens_per_epoch,
            n_epochs=n_epochs,
            sigma=sigma,
            attn_prior_weight=attn_prior_weight)

    def anneal_temperature(self, epoch):
        if self.pseudo_temp_anneal_epochs <= 0:
            return
        progress = min(epoch / self.pseudo_temp_anneal_epochs, 1.0)
        self.prong1.temperature = (
            self.pseudo_temperature_init -
            progress * (self.pseudo_temperature_init - self.pseudo_temp_min))

    def forward(self, boundary_probs, encoder_h, regime_attention,
                epoch_labels, epoch=0):
        self.anneal_temperature(epoch)
        pseudo_loss = self.prong1(boundary_probs, encoder_h)
        prior_results = self.prong2(
            boundary_probs, regime_attention, epoch_labels)

        # Prong 3: Variance penalty (anti collapse)
        # Negative variance = penalize constant outputs
        # Computed per sample then averaged
        var_per_sample = boundary_probs.var(dim=-1)  # [B]
        variance_loss = -var_per_sample.mean()

        total = (self.pseudo_weight * pseudo_loss +
                 self.prior_weight * prior_results['total'] +
                 self.variance_weight * variance_loss)
        return {
            'acbl_total': total,
            'pseudo_boundary_loss': pseudo_loss,
            'boundary_target_loss': prior_results['boundary_target_loss'],
            'attention_prior_loss': prior_results['attention_prior_loss'],
            'variance_loss': variance_loss,
            'boundary_variance': var_per_sample.mean(),
            'n_transitions': prior_results['n_transitions'],
            'pseudo_temperature': self.prong1.temperature,
        }

def forward_with_intermediates(model, x, detach_boundaries=False):
    """
    Drop in replacement for model.forward() that exposes:
      encoder_h:        embeddings before boundary detection
      boundary_probs:   boundary head output
      regime_attention:  attention weights from last transformer block

    When detach_boundaries=True, the boundaries are detached before
    entering regime attention. This means classification loss CANNOT
    send gradients back to the boundary head. The boundary head only
    receives gradients from ACBL losses. This prevents classification
    from crushing boundary diversity during the formation phase.

    Follows the exact flow from tier 2 MultiResContrastiveNeuroState.
    """
    B, N, C, T = x.shape

    epoch_embs = []
    for i in range(N):
        emb = model.mr_encoder(x[:, i])
        emb = emb + model.epoch_embed[:, i]
        epoch_embs.append(emb)

    full_seq = torch.cat(epoch_embs, dim=1)
    full_seq = model.pos_drop(full_seq + model.pos_embed)

    # This is encoder_h: before boundary detection
    encoder_h = full_seq

    cp_out = model.changepoint_module(full_seq)
    boundaries = cp_out['boundaries']
    boundary_loss = cp_out['boundary_loss']

    # Gradient isolation: detach boundaries from classification path
    # so only ACBL losses can update the boundary head
    boundaries_for_attn = boundaries.detach() if detach_boundaries else boundaries

    # Run transformer blocks, capture attention from last block
    regime_attention = None
    for i, block in enumerate(model.blocks):
        if i == len(model.blocks) - 1:
            full_seq, attn_w = block(
                full_seq, boundaries_for_attn, return_attention=True)
            regime_attention = attn_w
        else:
            full_seq = block(full_seq, boundaries_for_attn)

    full_seq = model.norm(full_seq)
    tpe = model.tokens_per_epoch
    start = tpe * (N // 2)
    end = start + tpe
    pooled = full_seq[:, start:end, :].mean(dim=1)
    logits = model.head(pooled)

    return {
        'logits': logits,
        'boundary_loss': boundary_loss,
        'boundaries': boundaries,
        'encoder_h': encoder_h,
        'boundary_probs': boundaries,
        'regime_attention': regime_attention,
    }

def create_acbl_dataloaders(h5_path, batch_size=16, seed=42):
    """Subject level split with 3 epoch label sequences."""
    h5_path = Path(h5_path)
    rng = np.random.RandomState(seed)

    with h5py.File(h5_path, 'r') as f:
        labels = f['labels'][:]
        subject_ids = f['subject_ids'][:]

    if isinstance(subject_ids[0], (bytes, np.bytes_)):
        subject_ids = np.array([
            s.decode() if isinstance(s, bytes) else s
            for s in subject_ids])

    unique_subjects = np.unique(subject_ids)
    n_subjects = len(unique_subjects)
    rng.shuffle(unique_subjects)

    n_train = int(0.7 * n_subjects)
    n_val = int(0.15 * n_subjects)

    train_subj = set(unique_subjects[:n_train])
    val_subj = set(unique_subjects[n_train:n_train + n_val])
    test_subj = set(unique_subjects[n_train + n_val:])

    train_idx = np.where(np.isin(subject_ids, list(train_subj)))[0]
    val_idx = np.where(np.isin(subject_ids, list(val_subj)))[0]
    test_idx = np.where(np.isin(subject_ids, list(test_subj)))[0]

    print(f"Split: {len(train_subj)} train, {len(val_subj)} val, "
          f"{len(test_subj)} test subjects")
    for name, idx in [('Train', train_idx), ('Val', val_idx),
                      ('Test', test_idx)]:
        dist = dict(zip(*np.unique(labels[idx], return_counts=True)))
        print(f"  {name}: {len(idx)} epochs, {dist}")

    train_ds = MultiEpochDataset(
        h5_path, indices=train_idx, context_size=1)
    val_ds = MultiEpochDataset(
        h5_path, indices=val_idx, context_size=1)
    test_ds = MultiEpochDataset(
        h5_path, indices=test_idx, context_size=1)

    train_ld = DataLoader(
        train_ds, batch_size, sampler=train_ds.get_sampler(),
        num_workers=0, drop_last=True)
    val_ld = DataLoader(
        val_ds, batch_size, shuffle=False, num_workers=0)
    test_ld = DataLoader(
        test_ds, batch_size, shuffle=False, num_workers=0)

    return train_ld, val_ld, test_ld


In [3]:
# CHB MIT DATA DEFINITIONS

# Copied from the seizure onset evaluation notebook so that the multi
# seed runs reproduce the split behind the reported CHB-MIT results.
# Two deliberate changes from the original: __getitem__ returns the
# signal under the key 'epoch' to match MultiEpochDataset, and the
# unused 'indices' and 'subject' entries are dropped because the default
# collate handles them awkwardly and nothing downstream reads them.


def load_chbmit_data(h5_path):
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() for s in f['subject_ids'][:]])
        sfreq = f.attrs['sfreq']
    print(f"Loaded: {epochs.shape[0]} epochs, {epochs.shape[1]} ch, "
          f"{sfreq} Hz")
    print(f"  Seizure: {np.sum(labels == 1)}, "
          f"Normal: {np.sum(labels == 0)}")
    subjects = np.unique(subject_ids)
    subj_sz = {}
    for s in subjects:
        subj_sz[s] = int(np.sum(labels[subject_ids == s] == 1))
    for s in sorted(subj_sz, key=subj_sz.get, reverse=True):
        print(f"    {s}: {subj_sz[s]} seizure epochs")
    return epochs, labels, subject_ids, subj_sz


def round_robin_split(subj_sz, seed=42):
    """Subjects sorted by seizure count, then dealt round robin.

    The split is deterministic and does not depend on seed, so it stays
    fixed across the multi seed runs by construction.
    """
    sorted_s = sorted(subj_sz.keys(), key=lambda s: subj_sz[s],
                      reverse=True)
    splits = {'train': [], 'val': [], 'test': []}
    pattern = ['train', 'train', 'val', 'test'] * 4
    for i, s in enumerate(sorted_s):
        splits[pattern[i % len(pattern)]].append(s)
    for name, subjs in splits.items():
        total = sum(subj_sz[s] for s in subjs)
        print(f"  {name}: {subjs} ({total} seizure epochs)")
    return splits


class CHBMITOnsetDataset(Dataset):
    def __init__(self, epochs, labels, subject_ids, subject_list,
                 n_context=3):
        mask = np.isin(subject_ids, subject_list)
        self.epochs = epochs[mask]
        self.labels = labels[mask]
        self.subject_ids = subject_ids[mask]
        self.n_ctx = n_context
        self.windows = []
        for subj in subject_list:
            subj_mask = self.subject_ids == subj
            subj_idx = np.where(subj_mask)[0]
            if len(subj_idx) < n_context:
                continue
            for start in range(len(subj_idx) - n_context + 1):
                group = subj_idx[start:start + n_context]
                if np.all(np.diff(group) == 1):
                    center = n_context // 2
                    center_label = self.labels[group[center]]
                    epoch_labels = [int(self.labels[g]) for g in group]
                    self.windows.append({
                        'indices': group,
                        'label': int(center_label),
                        'epoch_labels': epoch_labels,
                        'subject': subj,
                    })
        win_labels = [w['label'] for w in self.windows]
        print(f"  {len(self.windows)} windows "
              f"({sum(win_labels)} seizure, "
              f"{len(win_labels) - sum(win_labels)} normal)")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        x = np.stack([self.epochs[i] for i in w['indices']], axis=0)
        return {
            'epoch': torch.tensor(x, dtype=torch.float32),
            'label': torch.tensor(w['label'], dtype=torch.long),
            'epoch_labels': torch.tensor(w['epoch_labels'],
                                         dtype=torch.long),
        }


def get_class_weighted_sampler(dataset):
    labels = [w['label'] for w in dataset.windows]
    counts = np.bincount(labels)
    weights = 1.0 / counts
    sample_w = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, len(sample_w))


_CHB_CACHE = {}


def create_chbmit_dataloaders(h5_path, batch_size=32, seed=42,
                              n_context=3):
    """Round robin split loaders matching the reported CHB-MIT run.

    The full array is cached at module level because load_chbmit_data
    reads everything into memory; without the cache it would be re read
    for every seed and dominate the sweep runtime.
    """
    key = str(h5_path)
    if key not in _CHB_CACHE:
        _CHB_CACHE[key] = load_chbmit_data(Path(h5_path))
    epochs, labels, subject_ids, subj_sz = _CHB_CACHE[key]

    splits = round_robin_split(subj_sz, seed)
    train_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                  splits['train'], n_context)
    val_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                splits['val'], n_context)
    test_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                 splits['test'], n_context)

    train_ld = DataLoader(train_ds, batch_size=batch_size,
                          sampler=get_class_weighted_sampler(train_ds),
                          num_workers=0)
    val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                        num_workers=0)
    test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                         num_workers=0)
    return train_ld, val_ld, test_ld


In [4]:
# SEEDING

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [5]:
# CONFIG

SLEEP_EDF_CONFIG = {
    'name': 'sleep_edf',
    'h5_path': 'data/processed/sleep_edf_processed.h5',
    'n_channels': 3,
    'n_samples': 3000,
    'n_classes': 5,
    'batch_size': 16,
    'split_seed': 42,
    'embed_dim': 128,
    'n_layers': 4,
    'dropout': 0.1,
    'n_intra': 4,
    'n_inter': 2,
    'n_cross': 2,
    'contrast_scales': (1, 4, 16),
    'cp_hidden': 64,
    'n_context_epochs': 3,
    'n_epochs': 50,
    'warmup_epochs': 3,
    'formation_epochs': 12,
    'patience': 12,
    'acbl_weight': 0.3,
    'cls_weight': 1.0,
    'pseudo_weight': 0.3,
    'prior_weight': 0.5,
    'sigma': 5.0,
    'pseudo_temperature': 2.0,
    'pseudo_temp_min': 0.5,
    'pseudo_temp_anneal_epochs': 15,
    'attn_prior_weight': 0.5,
    'variance_weight': 1.0,
    'lr_other': 1e-4,
    'lr_boundary': 3e-4,
    'wd_other': 1e-4,
    'wd_boundary': 1e-5,
    'binary': False,
    'loader': 'sleep_edf',
}

# CHB-MIT uses the combined per subject file, a 17 channel montage and
# batch size 32, and is loaded through the round robin split rather than
# the random subject shuffle used for Sleep-EDF.
CHB_MIT_CONFIG = dict(SLEEP_EDF_CONFIG)
CHB_MIT_CONFIG.update({
    'name': 'chb_mit',
    'h5_path': 'data/processed/chbmit_combined_seizure_detection.h5',
    'n_channels': 17,
    'n_classes': 2,
    'batch_size': 32,
    'binary': True,
    'loader': 'chbmit',
    'single_param_group': True,
    'class_weighted_loss': True,
})


In [6]:
# QUIET EVALUATION

def evaluate_quiet(model, test_ld, device='cuda', binary=False):
    """Test metrics computed from whatever checkpoint is currently loaded.

    Classification metrics and boundary statistics come from the SAME
    model state, which removes the checkpoint mismatch between the
    validation selected weights and the final epoch boundary std.
    """
    model.eval()
    model.to(device)
    all_preds, all_labels, all_probs, all_bounds = [], [], [], []
    with torch.no_grad():
        for batch in test_ld:
            x = batch['epoch'].to(device)
            y = batch['label']
            out = model(x, return_boundaries=True)
            logits = out['logits']
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
            if binary:
                probs = torch.softmax(logits, dim=1)[:, 1]
                all_probs.extend(probs.cpu().numpy())
            all_bounds.append(out['boundaries'].cpu().numpy())

    preds = np.array(all_preds)
    labels = np.array(all_labels)
    bounds = np.concatenate(all_bounds)

    res = {
        'accuracy': float(accuracy_score(labels, preds)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro',
                                   zero_division=0)),
        'kappa': float(cohen_kappa_score(labels, preds)),
        'boundary_mean': float(bounds.mean()),
        'boundary_std': float(bounds.std()),
        'boundary_max': float(bounds.max()),
        'boundary_frac_05': float((bounds > 0.5).mean()),
    }
    if binary:
        res['auroc'] = float(roc_auc_score(labels, np.array(all_probs)))
        res['sensitivity'] = float(recall_score(labels, preds, pos_label=1,
                                                zero_division=0))
        res['specificity'] = float(recall_score(labels, preds, pos_label=0,
                                                zero_division=0))
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
        res['confusion'] = [int(tn), int(fp), int(fn), int(tp)]
    return res


In [7]:
# SINGLE RUN

def train_one(cfg, seed, device='cuda', verbose=False):
    """One full three phase ACBL run at a given training seed.

    The data split is controlled by cfg['split_seed'] and held fixed so
    that variance across seeds reflects initialisation and batch order,
    not a different set of test subjects.

    Optimiser and loss follow the original per dataset recipes:
    Sleep-EDF uses two parameter groups with a higher boundary learning
    rate and an unweighted loss, CHB-MIT uses a single group and a class
    weighted loss to handle the 3.2 percent seizure prevalence.
    """
    set_all_seeds(seed)

    if cfg.get('loader', 'sleep_edf') == 'chbmit':
        train_ld, val_ld, test_ld = create_chbmit_dataloaders(
            cfg['h5_path'], batch_size=cfg['batch_size'],
            seed=cfg['split_seed'],
            n_context=cfg['n_context_epochs'])
    else:
        train_ld, val_ld, test_ld = create_acbl_dataloaders(
            Path(cfg['h5_path']), batch_size=cfg['batch_size'],
            seed=cfg['split_seed'])

    model = MultiResContrastiveNeuroState(
        n_channels=cfg['n_channels'], n_samples=cfg['n_samples'],
        n_classes=cfg['n_classes'], embed_dim=cfg['embed_dim'],
        n_layers=cfg['n_layers'], dropout=cfg['dropout'],
        n_intra=cfg['n_intra'], n_inter=cfg['n_inter'],
        n_cross=cfg['n_cross'], contrast_scales=cfg['contrast_scales'],
        cp_hidden=cfg['cp_hidden'],
        n_context_epochs=cfg['n_context_epochs']).to(device)

    acbl_loss = ACBLLoss(
        tokens_per_epoch=model.tokens_per_epoch,
        n_epochs=model.n_context,
        pseudo_weight=cfg['pseudo_weight'],
        prior_weight=cfg['prior_weight'],
        sigma=cfg['sigma'],
        pseudo_temperature=cfg['pseudo_temperature'],
        pseudo_temp_min=cfg['pseudo_temp_min'],
        pseudo_temp_anneal_epochs=cfg['pseudo_temp_anneal_epochs'],
        attn_prior_weight=cfg['attn_prior_weight'],
        variance_weight=cfg['variance_weight']).to(device)

    if cfg.get('single_param_group', False):
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=cfg['lr_other'],
                                      weight_decay=cfg['wd_other'])
    else:
        boundary_params, other_params = [], []
        for pname, p in model.named_parameters():
            if 'changepoint' in pname:
                boundary_params.append(p)
            else:
                other_params.append(p)
        optimizer = torch.optim.AdamW([
            {'params': other_params, 'lr': cfg['lr_other'],
             'weight_decay': cfg['wd_other']},
            {'params': boundary_params, 'lr': cfg['lr_boundary'],
             'weight_decay': cfg['wd_boundary']},
        ])

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    if cfg.get('class_weighted_loss', False):
        train_labels = [w['label'] for w in train_ld.dataset.windows]
        n0 = sum(1 for l in train_labels if l == 0)
        n1 = sum(1 for l in train_labels if l == 1)
        cw = torch.tensor([len(train_labels) / (2 * n0),
                           len(train_labels) / (2 * n1)],
                          dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
    else:
        criterion = nn.CrossEntropyLoss()

    warmup = cfg['warmup_epochs']
    formation = cfg['formation_epochs']
    min_epochs = warmup + formation + 1
    best_val_cls = float('inf')
    best_state = None
    best_epoch = -1
    wait = 0
    trace = []

    for epoch in range(cfg['n_epochs']):
        if epoch < warmup:
            phase = 'warmup'
        elif epoch < warmup + formation:
            phase = 'form'
        else:
            phase = 'full'
        use_acbl = epoch >= warmup
        detach_bnd = (phase == 'form')

        if epoch == warmup + formation:
            wait = 0
            best_val_cls = float('inf')

        model.train()
        tr_cls, tr_n, tr_correct = 0.0, 0, 0
        for batch in train_ld:
            x = batch['epoch'].to(device)
            y = batch['label'].to(device)
            elabels = batch['epoch_labels'].to(device)
            optimizer.zero_grad()
            if use_acbl:
                out = forward_with_intermediates(
                    model, x, detach_boundaries=detach_bnd)
                cls_loss = criterion(out['logits'], y)
                acbl_results = acbl_loss(
                    out['boundary_probs'], out['encoder_h'],
                    out['regime_attention'], elabels, epoch)
                loss = (cfg['cls_weight'] * cls_loss +
                        cfg['acbl_weight'] * acbl_results['acbl_total'] +
                        out['boundary_loss'])
            else:
                out = model(x, return_boundaries=True)
                cls_loss = criterion(out['logits'], y)
                loss = cls_loss + out['boundary_loss']
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_cls += cls_loss.item() * len(y)
            tr_correct += (out['logits'].argmax(1) == y).sum().item()
            tr_n += len(y)

        model.eval()
        v_cls, v_n, v_correct = 0.0, 0, 0
        v_preds, v_labels, v_bounds = [], [], []
        with torch.no_grad():
            for batch in val_ld:
                x = batch['epoch'].to(device)
                y = batch['label'].to(device)
                out = model(x, return_boundaries=True)
                cls_loss = criterion(out['logits'], y)
                v_cls += cls_loss.item() * len(y)
                v_correct += (out['logits'].argmax(1) == y).sum().item()
                v_n += len(y)
                v_preds.extend(out['logits'].argmax(1).cpu().numpy())
                v_labels.extend(y.cpu().numpy())
                v_bounds.append(out['boundaries'].cpu().numpy())

        val_cls_loss = v_cls / v_n
        bounds_arr = np.concatenate(v_bounds)
        scheduler.step(val_cls_loss)

        trace.append({
            'epoch': epoch,
            'phase': phase,
            'train_cls': tr_cls / tr_n,
            'val_cls': val_cls_loss,
            'val_acc': v_correct / v_n,
            'bnd_mean': float(bounds_arr.mean()),
            'bnd_std': float(bounds_arr.std()),
        })

        if val_cls_loss < best_val_cls:
            best_val_cls = val_cls_loss
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        if verbose:
            print(f"  seed {seed} ep {epoch:3d} [{phase:6s}] "
                  f"val_cls={val_cls_loss:.4f} "
                  f"val_acc={v_correct/v_n:.4f} "
                  f"bnd_std={bounds_arr.std():.4f}")

        # Early stopping is enabled only after the full phase begins, so
        # a run cannot terminate during formation and be evaluated with
        # the boundary head still detached.
        if wait >= cfg['patience'] and epoch >= min_epochs:
            break

    if best_state:
        model.load_state_dict(best_state)
        model.to(device)

    ckpt_dir = Path('models/multiseed')
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(),
               ckpt_dir / f"{cfg['name']}_seed{seed}.pt")

    results = evaluate_quiet(model, test_ld, device=device,
                             binary=cfg['binary'])
    results['best_epoch'] = int(best_epoch)
    results['seed'] = int(seed)
    results['split_seed'] = int(cfg['split_seed'])
    results['stopped_epoch'] = int(trace[-1]['epoch'])
    return results, trace


In [8]:
# MULTI SEED LOOP

def run_seeds(cfg, seeds=(42, 43, 44, 45, 46), device='cuda',
              out_dir='models/multiseed', tag=None, verbose=False):
    """Run every seed, saving after each so a session restart never
    loses completed work. Re run the same call to resume."""
    tag = tag or cfg['name']
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / f'{tag}_results.json'

    completed = {}
    if results_path.exists():
        completed = {int(r['seed']): r
                     for r in json.loads(results_path.read_text())}
        print(f"Resuming: {sorted(completed)} already done")

    for seed in seeds:
        if seed in completed:
            print(f"Seed {seed} already complete, skipping")
            continue
        print(f"Running {tag} seed {seed}")
        t0 = time.time()
        try:
            res, trace = train_one(cfg, seed, device=device,
                                   verbose=verbose)
        except Exception as exc:
            print(f"Seed {seed} failed: {exc}")
            continue
        res['runtime_sec'] = round(time.time() - t0, 1)
        completed[seed] = res
        (out_dir / f'{tag}_trace_seed{seed}.json').write_text(
            json.dumps(trace, indent=2))
        results_path.write_text(
            json.dumps([completed[s] for s in sorted(completed)], indent=2))
        summary = (f"  acc={res['accuracy']:.4f} "
                   f"kappa={res['kappa']:.4f} "
                   f"bnd_std={res['boundary_std']:.4f} "
                   f"best_ep={res['best_epoch']} "
                   f"({res['runtime_sec']:.0f}s)")
        if cfg['binary']:
            summary += f" auroc={res['auroc']:.4f}"
        print(summary)

    return [completed[s] for s in sorted(completed)]


In [9]:
# AGGREGATION

def aggregate(results, keys=None):
    """Mean and sample std across seeds for each metric."""
    if not results:
        return {}
    if keys is None:
        keys = [k for k, v in results[0].items()
                if isinstance(v, (int, float)) and k not in
                ('seed', 'split_seed', 'best_epoch', 'stopped_epoch',
                 'runtime_sec')]
    agg = {}
    for k in keys:
        vals = np.array([r[k] for r in results if k in r], dtype=float)
        agg[k] = {
            'mean': float(vals.mean()),
            'std': float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
            'n': int(len(vals)),
            'values': [float(v) for v in vals],
        }
    return agg

def fmt(agg, key, dp=3):
    """Format one metric as mean plus or minus std for a LaTeX cell."""
    if key not in agg:
        return '--'
    m, s = agg[key]['mean'], agg[key]['std']
    return f"{m:.{dp}f} $\\pm$ {s:.{dp}f}"

def latex_row(label, agg, keys, dp=3):
    cells = ' & '.join(fmt(agg, k, dp) for k in keys)
    return f"{label} & {cells} \\\\"

def print_summary(results, cfg):
    agg = aggregate(results)
    n = len(results)
    print(f"\n{cfg['name']} over {n} seeds "
          f"(split_seed {cfg['split_seed']} fixed)")
    for k in sorted(agg):
        a = agg[k]
        print(f"  {k:22s} {a['mean']:.4f} +/- {a['std']:.4f}")
    if cfg['binary']:
        keys = ['accuracy', 'auroc', 'kappa', 'sensitivity', 'boundary_std']
    else:
        keys = ['accuracy', 'f1_macro', 'kappa', 'boundary_std']
    print("\nLaTeX row:")
    print(latex_row(cfg['name'], agg, keys))
    return agg


In [10]:
# CONDITIONAL BOUNDARY ANALYSIS

# The pooled boundary std is diluted on tasks where most windows contain
# no within-window transition. On CHB-MIT only 53 of 1496 test windows
# contain a seizure, so the pooled figure is dominated by windows where a
# flat boundary is the correct response, exactly as on TUAB. This splits
# the statistic by window type and tests whether the two populations
# differ.


def boundary_by_window_type(model, loader, device='cuda'):
    """Boundary statistics split by within-window label transition.

    Reports the pooled std within each subgroup, which is the quantity
    the paper tabulates, and the distribution of per-window stds, which
    is what the transition-marking claim is actually about.
    """
    model.eval()
    model.to(device)
    trans_tokens, flat_tokens = [], []
    trans_win, flat_win = [], []
    trans_peak, flat_peak = [], []

    with torch.no_grad():
        for batch in loader:
            x = batch['epoch'].to(device)
            el = batch['epoch_labels'].numpy()
            out = model(x, return_boundaries=True)
            b = out['boundaries'].cpu().numpy()
            has_trans = el.max(axis=1) != el.min(axis=1)
            for i in range(b.shape[0]):
                if has_trans[i]:
                    trans_tokens.append(b[i])
                    trans_win.append(float(b[i].std()))
                    trans_peak.append(float(b[i].max()))
                else:
                    flat_tokens.append(b[i])
                    flat_win.append(float(b[i].std()))
                    flat_peak.append(float(b[i].max()))

    res = {
        'n_transition': len(trans_win),
        'n_flat': len(flat_win),
    }
    if trans_tokens:
        arr = np.concatenate(trans_tokens)
        res['pooled_std_transition'] = float(arr.std())
        res['mean_window_std_transition'] = float(np.mean(trans_win))
        res['mean_peak_transition'] = float(np.mean(trans_peak))
    if flat_tokens:
        arr = np.concatenate(flat_tokens)
        res['pooled_std_flat'] = float(arr.std())
        res['mean_window_std_flat'] = float(np.mean(flat_win))
        res['mean_peak_flat'] = float(np.mean(flat_peak))

    if len(trans_win) >= 3 and len(flat_win) >= 3:
        u, p = mannwhitneyu(trans_win, flat_win, alternative='greater')
        res['mannwhitney_u'] = float(u)
        res['p_value'] = float(p)
        pooled_sd = np.sqrt((np.var(trans_win, ddof=1) +
                             np.var(flat_win, ddof=1)) / 2)
        if pooled_sd > 0:
            res['cohens_d'] = float(
                (np.mean(trans_win) - np.mean(flat_win)) / pooled_sd)
    res['per_window_std_transition'] = trans_win
    res['per_window_std_flat'] = flat_win
    return res


def run_conditional_analysis(cfg, seeds=(42, 43, 44, 45, 46),
                             device='cuda',
                             ckpt_dir='models/multiseed'):
    """Load each saved checkpoint and report the conditional statistics."""
    ckpt_dir = Path(ckpt_dir)
    if cfg.get('loader', 'sleep_edf') == 'chbmit':
        _, _, test_ld = create_chbmit_dataloaders(
            cfg['h5_path'], batch_size=cfg['batch_size'],
            seed=cfg['split_seed'], n_context=cfg['n_context_epochs'])
    else:
        _, _, test_ld = create_acbl_dataloaders(
            Path(cfg['h5_path']), batch_size=cfg['batch_size'],
            seed=cfg['split_seed'])

    per_seed = []
    for seed in seeds:
        ckpt = ckpt_dir / f"{cfg['name']}_seed{seed}.pt"
        if not ckpt.exists():
            print(f"missing checkpoint for seed {seed}: {ckpt}")
            continue
        model = MultiResContrastiveNeuroState(
            n_channels=cfg['n_channels'], n_samples=cfg['n_samples'],
            n_classes=cfg['n_classes'], embed_dim=cfg['embed_dim'],
            n_layers=cfg['n_layers'], dropout=cfg['dropout'],
            n_intra=cfg['n_intra'], n_inter=cfg['n_inter'],
            n_cross=cfg['n_cross'],
            contrast_scales=cfg['contrast_scales'],
            cp_hidden=cfg['cp_hidden'],
            n_context_epochs=cfg['n_context_epochs'])
        model.load_state_dict(torch.load(ckpt, map_location='cpu'))
        r = boundary_by_window_type(model, test_ld, device=device)
        r['seed'] = seed
        per_seed.append(r)
        print(f"seed {seed}: "
              f"transition n={r['n_transition']} "
              f"pooled_std={r.get('pooled_std_transition', float('nan')):.4f} "
              f"win_std={r.get('mean_window_std_transition', float('nan')):.4f} "
              f"| flat n={r['n_flat']} "
              f"pooled_std={r.get('pooled_std_flat', float('nan')):.4f} "
              f"win_std={r.get('mean_window_std_flat', float('nan')):.4f} "
              f"| p={r.get('p_value', float('nan')):.4g}")

    if per_seed:
        def agg(key):
            vals = [r[key] for r in per_seed if key in r]
            if not vals:
                return float('nan'), float('nan')
            return float(np.mean(vals)), float(
                np.std(vals, ddof=1) if len(vals) > 1 else 0.0)

        print(f"\n{cfg['name']} conditional boundary std over "
              f"{len(per_seed)} seeds")
        for key in ('pooled_std_transition', 'pooled_std_flat',
                    'mean_window_std_transition', 'mean_window_std_flat',
                    'mean_peak_transition', 'mean_peak_flat',
                    'cohens_d'):
            m, s = agg(key)
            print(f"  {key:30s} {m:.4f} +/- {s:.4f}")
        ps = [r['p_value'] for r in per_seed if 'p_value' in r]
        if ps:
            print(f"  {'p_value (max across seeds)':30s} {max(ps):.4g}")

        out = ckpt_dir / f"{cfg['name']}_conditional.json"
        slim = [{k: v for k, v in r.items()
                 if not k.startswith('per_window')} for r in per_seed]
        out.write_text(json.dumps(slim, indent=2))
        print(f"\nSaved to {out}")
    return per_seed

## Sanity check

Confirm seed 42 reproduces the submitted numbers before launching the full sweep.

In [11]:
# SANITY CHECK BEFORE THE FULL SWEEP

# Confirm seed 42 reproduces the submitted numbers before spending GPU
# hours on the whole sweep. Sleep-EDF should land near Acc 0.718 and
# Kappa 0.621, and the printed parameter count is 1.00M on the 3 channel
# montage (the 1.07M quoted in the paper is the 17 channel model).

# results_42, trace_42 = train_one(SLEEP_EDF_CONFIG, seed=42,
#                                  device='cuda', verbose=True)
# print(json.dumps({k: v for k, v in results_42.items()
#                   if not isinstance(v, list)}, indent=2))

# for k, target in {'accuracy': 0.718, 'kappa': 0.621}.items():
#     got = results_42[k]
#     delta = abs(got - target)
#     flag = 'ok' if delta < 0.02 else 'CHECK: diverges from paper'
#     print(f"{k}: got {got:.4f}, paper {target:.3f}, "
#           f"delta {delta:.4f} {flag}")


## Headline sweeps

In [12]:
# SLEEP EDF MULTI SEED

sleep_results = run_seeds(SLEEP_EDF_CONFIG,
                          seeds=(42, 43, 44, 45, 46),
                          device='cuda',
                          out_dir='models/multiseed',
                          tag='sleep_edf')
sleep_agg = print_summary(sleep_results, SLEEP_EDF_CONFIG)


Resuming: [42, 43, 44, 45, 46] already done
Seed 42 already complete, skipping
Seed 43 already complete, skipping
Seed 44 already complete, skipping
Seed 45 already complete, skipping
Seed 46 already complete, skipping

sleep_edf over 5 seeds (split_seed 42 fixed)
  accuracy               0.6976 +/- 0.0126
  balanced_accuracy      0.6356 +/- 0.0257
  boundary_frac_05       0.0000 +/- 0.0000
  boundary_max           0.2604 +/- 0.0083
  boundary_mean          0.2066 +/- 0.0069
  boundary_std           0.0260 +/- 0.0029
  f1_macro               0.6116 +/- 0.0187
  kappa                  0.5959 +/- 0.0184

LaTeX row:
sleep_edf & 0.698 $\pm$ 0.013 & 0.612 $\pm$ 0.019 & 0.596 $\pm$ 0.018 & 0.026 $\pm$ 0.003 \\


In [13]:
# CHB MIT MULTI SEED

# The expected split is train chb15/12/01/03/14/17, val chb08/10/22,
# test chb05/20/19, with 1496 test windows of which 53 are seizure.
# A different subject list means the combined file does not match the
# run that produced AUROC 0.940 and should be resolved first.

chb_results = run_seeds(CHB_MIT_CONFIG,
                        seeds=(42, 43, 44, 45, 46),
                        device='cuda',
                        out_dir='models/multiseed',
                        tag='chb_mit')
chb_agg = print_summary(chb_results, CHB_MIT_CONFIG)


Running chb_mit seed 42
Loaded: 9639 epochs, 17 ch, 100 Hz
  Seizure: 312, Normal: 9327
    chb15: 73 seizure epochs
    chb12: 54 seizure epochs
    chb08: 35 seizure epochs
    chb05: 34 seizure epochs
    chb01: 24 seizure epochs
    chb03: 22 seizure epochs
    chb10: 19 seizure epochs
    chb20: 13 seizure epochs
    chb14: 12 seizure epochs
    chb17: 10 seizure epochs
    chb22: 9 seizure epochs
    chb19: 7 seizure epochs
  train: ['chb15', 'chb12', 'chb01', 'chb03', 'chb14', 'chb17'] (195 seizure epochs)
  val: ['chb08', 'chb10', 'chb22'] (63 seizure epochs)
  test: ['chb05', 'chb20', 'chb19'] (54 seizure epochs)
  4671 windows (193 seizure, 4478 normal)
  3448 windows (63 seizure, 3385 normal)
  1496 windows (53 seizure, 1443 normal)
ContrastiveMultiRes: 1.07M params, 588 tokens
  acc=0.9171 kappa=0.3706 bnd_std=0.0080 best_ep=17 (3415s) auroc=0.9300
Running chb_mit seed 43
  train: ['chb15', 'chb12', 'chb01', 'chb03', 'chb14', 'chb17'] (195 seizure epochs)
  val: ['chb08', '

## Outputs for the paper

In [14]:
# CAMERA READY TABLE ROWS

rows = []
if 'sleep_agg' in dir():
    rows.append(latex_row('Sleep-EDF', sleep_agg,
                          ['accuracy', 'f1_macro', 'kappa',
                           'boundary_std']))
if 'chb_agg' in dir():
    rows.append(latex_row('CHB-MIT', chb_agg,
                          ['accuracy', 'auroc', 'kappa', 'sensitivity',
                           'boundary_std']))
for r in rows:
    print(r)

Path('models/multiseed').mkdir(parents=True, exist_ok=True)
Path('models/multiseed/latex_rows.txt').write_text('\n'.join(rows))
print('Saved to models/multiseed/latex_rows.txt')


Sleep-EDF & 0.698 $\pm$ 0.013 & 0.612 $\pm$ 0.019 & 0.596 $\pm$ 0.018 & 0.026 $\pm$ 0.003 \\
CHB-MIT & 0.901 $\pm$ 0.043 & 0.924 $\pm$ 0.014 & 0.356 $\pm$ 0.132 & 0.796 $\pm$ 0.047 & 0.014 $\pm$ 0.006 \\
Saved to models/multiseed/latex_rows.txt


In [15]:
# CHECKPOINT CONSISTENCY CHECK

# The paper reports CHB-MIT classification from the epoch 15 best
# validation checkpoint but boundary std 0.035 from the final epoch.
# evaluate_quiet takes both from the same restored checkpoint, so the
# boundary std below is the internally consistent number to report.

p = Path('models/multiseed/chb_mit_results.json')
if p.exists():
    res = json.loads(p.read_text())
    for r in res:
        print(f"seed {r['seed']}: best_epoch {r['best_epoch']}, "
              f"acc {r['accuracy']:.4f}, "
              f"auroc {r.get('auroc', float('nan')):.4f}, "
              f"bnd_std {r['boundary_std']:.4f}")
    stds = [r['boundary_std'] for r in res]
    print(f"\nboundary std across seeds: "
          f"{sum(stds) / len(stds):.4f} mean, min {min(stds):.4f}, "
          f"max {max(stds):.4f}")
    print("If this sits below 0.020, the threshold story needs the "
          "null distribution from the collapse notebook.")
else:
    print('Run the CHB-MIT sweep first')


seed 42: best_epoch 17, acc 0.9171, auroc 0.9300, bnd_std 0.0080
seed 43: best_epoch 20, acc 0.8690, auroc 0.9229, bnd_std 0.0121
seed 44: best_epoch 17, acc 0.8590, auroc 0.9005, bnd_std 0.0143
seed 45: best_epoch 17, acc 0.8937, auroc 0.9369, bnd_std 0.0242
seed 46: best_epoch 15, acc 0.9652, auroc 0.9286, bnd_std 0.0119

boundary std across seeds: 0.0141 mean, min 0.0080, max 0.0242
If this sits below 0.020, the threshold story needs the null distribution from the collapse notebook.


In [16]:
chb_cond = run_conditional_analysis(CHB_MIT_CONFIG)

  train: ['chb15', 'chb12', 'chb01', 'chb03', 'chb14', 'chb17'] (195 seizure epochs)
  val: ['chb08', 'chb10', 'chb22'] (63 seizure epochs)
  test: ['chb05', 'chb20', 'chb19'] (54 seizure epochs)
  4671 windows (193 seizure, 4478 normal)
  3448 windows (63 seizure, 3385 normal)
  1496 windows (53 seizure, 1443 normal)
ContrastiveMultiRes: 1.07M params, 588 tokens
seed 42: transition n=59 pooled_std=0.0080 win_std=0.0080 | flat n=1437 pooled_std=0.0080 win_std=0.0080 | p=0.9918
ContrastiveMultiRes: 1.07M params, 588 tokens
seed 43: transition n=59 pooled_std=0.0118 win_std=0.0117 | flat n=1437 pooled_std=0.0121 win_std=0.0119 | p=0.9997
ContrastiveMultiRes: 1.07M params, 588 tokens
seed 44: transition n=59 pooled_std=0.0141 win_std=0.0140 | flat n=1437 pooled_std=0.0143 win_std=0.0141 | p=0.9826
ContrastiveMultiRes: 1.07M params, 588 tokens
seed 45: transition n=59 pooled_std=0.0238 win_std=0.0225 | flat n=1437 pooled_std=0.0242 win_std=0.0224 | p=0.2067
ContrastiveMultiRes: 1.07M param

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>